# Tarea de clasificación

*Taller de proyecto - Grupo 4*

## Pregunta de investigación

> **¿Qué caracteriza a las escuelas de nivel socioeconómico bajo que alcanzan mejores logros de aprendizaje que otras escuelas en condiciones similares?**

Esta tarea de clasificación está diseñada para responder esa pregunta. Conviene explicitar cómo se traduce cada parte, porque de ahí salen todas las decisiones metodológicas del notebook.

| Parte de la pregunta | Cómo se traduce |
| :--- | :--- |
| *"escuelas de nivel socioeconómico bajo"* | Población: los 1.525 establecimientos de GSE bajo y medio-bajo definidos en el EDA. El GSE entra como variable de contexto, y al final se repite el análisis solo en GSE "Bajo" como prueba de robustez. |
| *"alcanzan mejores logros de aprendizaje"* | `target_bin`: superar el corte oficial de 252 puntos en Matemática II medio. |
| *"en condiciones similares"* | Separación explícita de los predictores en dos bloques: **contexto** (condiciones que se mantienen constantes) y **proceso** (rasgos del funcionamiento del establecimiento). |
| *"qué caracteriza"* | La respuesta es la **contribución incremental del bloque de proceso sobre el de contexto**, más la importancia por permutación dentro de ese bloque. |

### Por qué la separación en bloques es indispensable

Restringir la población a GSE bajo y medio-bajo es un control **grueso**: dentro de ese grupo queda mucha variación socioeconómica. Entre los establecimientos que superan el corte, solo el 25,5% son de GSE "Bajo", contra el 49,9% entre los que no lo superan. Es decir, una parte de lo que un clasificador aprendería a detectar es simplemente *"esta escuela es menos pobre"*, no *"esta escuela logra más en igualdad de condiciones"*.

La pregunta de investigación pide lo segundo. Un único modelo con los 18 predictores mezclados no permite separarlos, por eso el bloque de contexto se ajusta primero y se mide qué agrega el proceso **por sobre** él.

### Diseño del análisis

1. **Modelo de contexto**: cuánto del resultado se explica solo por condiciones socioeconómicas y tamaño.
2. **Modelo de contexto + proceso**: cuánto agregan los rasgos de funcionamiento. Esta ganancia incremental es la evidencia que responde la pregunta.
3. **Caracterización**: qué variables de proceso discriminan, por importancia por permutación.
4. **Análisis complementario del residuo**: comparación directa entre establecimientos que superan y que no alcanzan su puntaje esperado dado el contexto.

### Lo que este diseño NO puede afirmar

Se declara desde el inicio, porque son las limitaciones que condicionan la lectura de los resultados:

- **No hay causalidad.** Los datos son transversales (2024). La pregunta es descriptiva ("qué caracteriza"), y a eso se limitan las conclusiones. Nada aquí autoriza a afirmar que intervenir sobre una variable modifique el resultado.
- **Varianza de método común, no resuelta.** Los indicadores IDPS provienen del mismo cuestionario, aplicado a los mismos estudiantes, en la misma jornada que el SIMCE. Parte de su asociación con el puntaje puede deberse a esa simultaneidad y no a un rasgo estable del establecimiento. No hay en esta base una medición independiente que permita descartarlo.
- **Causalidad inversa plausible.** La asistencia y el clima laboral docente pueden ser consecuencia de un establecimiento que funciona bien, no su causa.

### Unidad de análisis y fuente

Establecimiento educacional (`rbd`). La entrada es `datos/base_tareas.parquet`, consolidada en `EDAyPreprocesamiento_01.ipynb` a partir de 8 fuentes. **Este notebook no modifica esa base ni ese notebook: solo la consume.**

### Ubicación en el proceso

Siguiendo CRISP-DM, el notebook de EDA cubrió *Business Understanding*, *Data Understanding* y *Data Preparation*. Esta tarea corresponde a **Modeling** y **Evaluation**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

sns.set_style("whitegrid")

# Semilla única para todo el notebook: cualquier procedimiento con aleatoriedad
# (particiones de CV, inicialización de modelos) la recibe explícitamente.
# Es la misma que se usó en el EDA y en la tarea de clustering.
SEED = 42

# Corte oficial de la Agencia de Calidad de la Educación entre los niveles
# de desempeño "Insuficiente" y "Elemental" en Matemática II medio.
CORTE = 252

## 1. Carga de datos

`base_tareas.parquet` guarda las variables categóricas solo como códigos numéricos (`cod_grupo`, `cod_depe2`, `cod_rural_rbd`). Se reconstruyen las etiquetas legibles con los mismos diccionarios usados en el EDA, porque las necesitamos tanto para los gráficos como para codificarlas correctamente más adelante: si dejáramos `cod_depe2` como entero, el modelo interpretaría que `SLEP (4)` es "cuatro veces" `Municipal (1)`, cuando la variable es puramente nominal.

In [ ]:
df = pd.read_parquet("datos/base_tareas.parquet")

map_gse   = {1: "Bajo", 2: "Medio bajo", 3: "Medio", 4: "Medio alto", 5: "Alto"}
map_depe2 = {1: "Municipal", 2: "Part. subvencionado", 3: "Part. pagado", 4: "SLEP"}
map_rural = {1: "Urbano", 2: "Rural"}

df["gse"]  = df["cod_grupo"].map(map_gse)
df["depe"] = df["cod_depe2"].map(map_depe2)
df["zona"] = df["cod_rural_rbd"].map(map_rural)

print(f"Establecimientos: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"\nGSE presentes en la base: {sorted(df['gse'].unique())}")
print(f"Etiquetas sin mapear (deben ser 0): {df[['gse', 'depe', 'zona']].isna().sum().sum()}")

df[["rbd", "gse", "depe", "zona", "prom_mate2m_rbd"]].head()

## 2. Construcción de la variable objetivo

El EDA ya decidió el target: las tres categorías originales ("Insuficiente", "Elemental", "Adecuado") se colapsan a **dos**, porque solo 15 establecimientos alcanzaban "Adecuado" y ningún modelo puede aprender una clase con esa frecuencia.

- `target_bin = 0` → **Insuficiente** (puntaje < 252)
- `target_bin = 1` → **Elemental o adecuado** (puntaje ≥ 252)

Definimos como clase positiva (`1`) la de **superar el corte**. Esta convención importa: es la clase minoritaria y la sustantivamente interesante (establecimientos vulnerables que logran resultados por sobre lo esperado), y determina cómo se leen la precisión, el recall y la curva PR más adelante.

In [ ]:
df["target_bin"] = (df["prom_mate2m_rbd"] >= CORTE).astype(int)
df["target_cat"] = np.where(df["target_bin"] == 1, "Elemental o adecuado", "Insuficiente")

balance = pd.DataFrame({
    "n": df["target_cat"].value_counts(),
    "pct": (df["target_cat"].value_counts(normalize=True) * 100).round(1),
})
balance.index.name = "nivel de desempeño"

print(f"Prevalencia de la clase positiva: {df['target_bin'].mean() * 100:.1f}%")
print(f"Razón de desbalance: 1 positivo por cada {(1 - df['target_bin'].mean()) / df['target_bin'].mean():.1f} negativos\n")
balance

### La banda de ambigüedad en torno al corte

Al dicotomizar una variable continua en un umbral, los casos **cercanos al corte** quedan con una etiqueta casi arbitraria: un establecimiento con 251 puntos y otro con 253 se etiquetan distinto, aunque su desempeño real es indistinguible dentro del error de medición de la prueba.

Cuantificamos esa banda ahora, al principio, porque será la explicación de buena parte del error irreducible del modelo. No es un defecto que haya que corregir: es una propiedad conocida del target que conviene documentar y revisar al final, cuando analicemos los errores.

In [ ]:
BANDA = 10  # puntos a cada lado del corte

en_banda = df["prom_mate2m_rbd"].between(CORTE - BANDA, CORTE + BANDA)
df["cerca_del_corte"] = en_banda

print(f"Establecimientos a ±{BANDA} puntos del corte: {en_banda.sum()} "
      f"({en_banda.mean() * 100:.1f}% de la base)")

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.hist(df["prom_mate2m_rbd"], bins=40, color="#5B95D6", edgecolor="white")
ax.axvspan(CORTE - BANDA, CORTE + BANDA, color="#b8860b", alpha=0.18,
           label=f"Banda de ambigüedad (±{BANDA} pts, n={en_banda.sum()})")
ax.axvline(CORTE, color="#b8860b", ls="--", lw=1.6, label=f"Corte = {CORTE}")

ax.set_title("Puntaje promedio de Matemática II medio y corte del target", loc="left")
ax.set_xlabel("Puntaje promedio SIMCE")
ax.set_ylabel("N° de establecimientos")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Fuga de información (*data leakage*)

Antes de definir la matriz de predictores hay que resolver el problema más importante del diseño. El target se construyó **como un umbral de `prom_mate2m_rbd`**, así que esa columna contiene el target por definición. Si la dejáramos entre los predictores, el modelo no aprendería nada sobre educación: solo recuperaría la regla `puntaje ≥ 252`.

Esto se llama fuga de información, y es el error más frecuente y más difícil de detectar en proyectos de clasificación, porque **se manifiesta como un resultado excelente**. Vale la pena verlo ocurrir una vez.

In [ ]:
# Demostración: el mismo modelo, la misma validación, cambiando solo si
# incluimos la variable a partir de la cual se definió el target.
cv_demo = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
modelo_demo = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=SEED),
)

for etiqueta, columnas in [
    ("CON fuga", ["prom_mate2m_rbd", "convivencia", "asistencia_2m"]),
    ("SIN fuga", ["convivencia", "asistencia_2m"]),
]:
    auc = cross_val_score(modelo_demo, df[columnas], df["target_bin"],
                          cv=cv_demo, scoring="roc_auc").mean()
    print(f"{etiqueta}: AUC 5-fold = {auc:.4f}   ({', '.join(columnas)})")

El AUC pasa de **0,75 a 0,9998**. Un modelo casi perfecto, obtenido sin ninguna capacidad predictiva real: simplemente le mostramos la respuesta.

La lección general es que **un resultado sospechosamente bueno debe llevar a auditar los predictores antes que a celebrar**. En este proyecto ya se aplicó el mismo criterio en el EDA para descartar `CLASIFICACION_SEP`: la Ley 20.248 clasifica a los establecimientos usando sus propios resultados educativos, por lo que era otra forma de fuga, más sutil pero igual de inválida.

Con eso claro, definimos las exclusiones.

In [ ]:
# --- Columnas que NO pueden entrar como predictores ---

# 1) Fuga directa: el target es una función determinista de esta variable.
cols_fuga = ["prom_mate2m_rbd"]

# 2) Identificador: no tiene contenido sustantivo. Si se dejara, un modelo de
#    árboles podría memorizar rbd individuales (sobreajuste puro).
cols_id = ["rbd"]

# 3) Derivados del target creados en este notebook.
cols_target = ["target_bin", "target_cat", "cerca_del_corte"]

# 4) Redundantes con las etiquetas legibles que acabamos de construir.
#    Se conservan las versiones de texto (gse, depe, zona) para codificarlas
#    como nominales, y se descartan los códigos numéricos equivalentes.
cols_codigos_redundantes = ["cod_grupo", "cod_depe2", "cod_rural_rbd"]

excluidas = cols_fuga + cols_id + cols_target + cols_codigos_redundantes

candidatas = [c for c in df.columns if c not in excluidas]

print(f"Columnas totales en la base:  {df.shape[1]}")
print(f"Excluidas por diseño:         {len(excluidas)}")
print(f"Candidatas a predictores:     {len(candidatas)}\n")
print("Excluidas:")
for c in excluidas:
    print(f"  - {c}")

---

### Estado del notebook

**Listo:** diseño de la matriz de predictores alineado con la pregunta de investigación (bloques contexto / proceso), pipeline de preprocesamiento sin fuga, protocolo de validación agrupado por comuna y métrica principal justificada contra sus pisos.

**Pendiente, mapeado a la rúbrica:**

| Sección | Rúbrica |
| :--- | :--- |
| 6. Algoritmos y ajuste de hiperparámetros | *Explicar algoritmos, datos e hiperparámetros* (10%) |
| 7. Diseño incremental: contexto → contexto + proceso | Responde la pregunta de investigación |
| 8. Histograma de scores y elección del umbral | *Histograma de scores y umbral de decisión* (parte del 20%) |
| 9. Métricas de efectividad al umbral elegido | *Métricas de efectividad* (parte del 20%) |
| 10. Revisión de casos: errores, banda de ambigüedad y escuelas que superan su expectativa | *Revisión de casos* (parte del 20%) |
| 11. Caracterización por importancia por permutación | Responde "qué caracteriza" |
| 12. Robustez: solo GSE "Bajo" y sensibilidad a la imputación docente | Limitaciones declaradas |

In [ ]:
# Tramos de la carrera docente:
#   0 = Sin información | 1 = Acceso | 2 = Inicial | 3 = Temprano
#   4 = Avanzado (estándar esperado) | 5 = Experto I | 6 = Experto II
df["pct_tramo_avanzado_sup"] = df[["pct_tramo_4", "pct_tramo_5", "pct_tramo_6"]].sum(axis=1)
df["pct_tramo_acceso_inicial"] = df[["pct_tramo_1", "pct_tramo_2"]].sum(axis=1)

# Años de servicio: los dos extremos (docentes nuevos y muy senior)
df["pct_anios_0a5"] = df["pct_anios_0-5"]
df["pct_anios_21mas"] = df[["pct_anios_21-30", "pct_anios_31+"]].sum(axis=1)

# Control de tamaño: total de docentes con horas de aula del establecimiento
df["n_docentes_total"] = df[[c for c in df.columns if c.startswith("n_tramo_")]].sum(axis=1)

vars_docentes = ["pct_tramo_avanzado_sup", "pct_tramo_acceso_inicial",
                 "pct_anios_0a5", "pct_anios_21mas", "n_docentes_total"]

# Verificación clave: las categorías que quedaron fuera (Sin información y
# Temprano; 6-10 y 11-20 años) actúan como referencia implícita, así que los
# grupos colapsados YA NO suman una constante. Se rompió la colinealidad perfecta.
suma_tramo = df[["pct_tramo_avanzado_sup", "pct_tramo_acceso_inicial"]].sum(axis=1)
suma_anios = df[["pct_anios_0a5", "pct_anios_21mas"]].sum(axis=1)
print(f"Suma de pct_tramo colapsados: rango [{suma_tramo.min():.1f}, {suma_tramo.max():.1f}]  (antes: 100 fijo)")
print(f"Suma de pct_anios colapsados: rango [{suma_anios.min():.1f}, {suma_anios.max():.1f}]  (antes: 100 fijo)")
print(f"\n20 columnas originales -> {len(vars_docentes)} variables temáticas")

df[vars_docentes].describe().round(2)

### 4.2 Colinealidad entre los predictores restantes

Resuelta la colinealidad *perfecta*, revisamos si queda colinealidad *alta*. Usamos dos herramientas complementarias:

- **Correlaciones por pares**: detecta redundancia entre dos variables.
- **VIF** (*Variance Inflation Factor*): detecta si una variable es predecible desde **todas las demás en conjunto**, algo que las correlaciones por pares no ven. Se calcula regresando cada predictor contra el resto: `VIF = 1 / (1 - R²)`. Un VIF de 10 significa que la varianza del coeficiente estimado está inflada 10 veces respecto de un escenario sin colinealidad. Regla práctica: sobre 5 merece atención, sobre 10 es problemático.

In [ ]:
from sklearn.linear_model import LinearRegression

# Set numérico a auditar (antes de decidir exclusiones)
num_auditar = [
    "autoestima_motiv", "convivencia", "participacion", "vida_saludable",
    "promedio_estandarizado_genero", "pct_acuerdo_promedio",
    "promedio_estandarizado_clima", "promedio_estandarizado_directivos",
    "ive_media", "asistencia_2m", "pct_preferentes",
] + vars_docentes

corr = df[num_auditar].corr()
pares = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack()
             .sort_values(key=abs, ascending=False))

print("Pares con |r| > 0.70:")
print(pares[abs(pares) > 0.70].round(3).to_string(), "\n")

# VIF. Se calcula sobre filas completas porque una regresión no admite NaN;
# es solo un diagnóstico, no forma parte del entrenamiento.
completas = df[num_auditar].dropna()
vif = {}
for col in num_auditar:
    otras = [c for c in num_auditar if c != col]
    r2 = LinearRegression().fit(completas[otras], completas[col]).score(completas[otras], completas[col])
    vif[col] = 1 / (1 - r2) if r2 < 1 else np.inf

tabla_vif = pd.Series(vif, name="VIF").sort_values(ascending=False).round(2).to_frame()
tabla_vif["alerta"] = np.select([tabla_vif["VIF"] > 10, tabla_vif["VIF"] > 5],
                                ["problemático", "atención"], default="")

print(f"VIF (diagnóstico sobre {len(completas)} filas completas):")
tabla_vif

**Lectura del diagnóstico y decisiones.**

**1. Redundancia genuina, que sí eliminamos.** `promedio_estandarizado_genero` y `pct_acuerdo_promedio` correlacionan **0,962**, con VIF de 17,6 y 17,0. No es sorpresa: el propio EDA verificó que el *factor score* y el promedio simple de los 6 ítems correlacionan prácticamente 1. Son la misma medición expresada de dos formas. Conservamos el **factor score** (`promedio_estandarizado_genero`), que es el indicador construido con análisis factorial policórico y el que se usó en los modelos multivariados del EDA, y descartamos el otro.

**2. Colinealidad moderada, que sí conservamos.** Los cuatro indicadores IDPS correlacionan entre 0,71 y 0,88 (VIF de 2,6 a 8,1). Es esperable: miden dimensiones de un mismo clima escolar. Aquí conviene separar dos cosas que suelen confundirse:

> La colinealidad **no degrada la capacidad predictiva**. Degrada la *interpretación de los coeficientes individuales* de un modelo lineal. Los modelos de árboles son casi indiferentes a ella.

Como buscamos predecir *y* interpretar, no los colapsamos, pero tomamos dos precauciones. Usaremos **regularización L2**, cuyo efecto es precisamente repartir el peso de forma estable entre predictores correlacionados, en vez de asignarlo de manera arbitraria. Y para interpretar no leeremos coeficientes crudos, sino **importancia por permutación**, que es robusta a este problema. Queda anotado como limitación explícita: el coeficiente individual de `participacion` no admite lectura aislada.

**3. Variables geográficas.** `cod_com_rbd` (319 comunas) y `cod_reg_rbd` (16 regiones) quedan **fuera de los predictores**, por razones distintas. La comuna será la variable de agrupación de la validación cruzada: incluirla como predictor anularía justamente el control que buscamos. Y la región, con 16 niveles nominales, agregaría 15 parámetros que capturan heterogeneidad no observada más que un mecanismo accionable, sobre una muestra con solo 357 casos positivos. Ambas se conservan en el DataFrame para agrupar los folds y para el análisis de errores.

### 4.3 Separación en bloques: contexto y proceso

Esta es la decisión que alinea el modelo con la pregunta de investigación, y no es una decisión estadística sino **conceptual**: hay que definir qué cuenta como "condición" y qué cuenta como "característica".

El criterio que usamos: una variable es **contexto** si el establecimiento **no la elige**, y **proceso** si describe cómo funciona. Bajo ese criterio adoptamos una definición estrecha de contexto — solo lo socioeconómico y el tamaño:

| Bloque | Variables | Rol en el análisis |
| :--- | :--- | :--- |
| **CONTEXTO** (5) | `ive_media`, `pct_preferentes`, `n_docentes_total`, `gse`, `zona` | Definen "condiciones similares". Se mantienen constantes. |
| **PROCESO** (13) | 4 IDPS, creencias de género, clima docente, coordinación directiva, `depe`, `asistencia_2m`, 4 de composición docente | Son la respuesta a "qué caracteriza". |

Tres advertencias sobre esta clasificación, que conviene tener presentes al leer los resultados:

**1. Si una variable de contexto resulta importante, eso no responde la pregunta.** Al contrario: confirma que es una condición. La pregunta se responde con lo que el bloque de proceso agrega **por sobre** el de contexto.

**2. La composición docente es discutible como "proceso".** Una escuela vulnerable no elige libremente qué docentes logra atraer y retener, así que `pct_tramo_avanzado_sup` podría defenderse como condición heredada. Lo tratamos como proceso porque la retención docente sí es en parte resultado de la gestión, pero es un supuesto y así queda registrado.

**3. `asistencia_2m` es ambigua entre proceso y resultado intermedio.** Es plausible que una escuela que funciona bien logre mejor asistencia, y también que la asistencia produzca aprendizaje. Al tratarla como proceso, parte de su aporte puede ser mediación y no característica. Es la variable a interpretar con más cautela.

Una consecuencia práctica: en el código, cambiar de criterio significa mover un nombre de una lista a otra. Vale la pena aprovecharlo y reportar en el informe qué tan sensibles son las conclusiones a esa reclasificación.

In [ ]:
# =========================================================================
# BLOQUE CONTEXTO — define "condiciones similares"
#
# Condiciones que el establecimiento no elige. Su función en el análisis NO es
# explicar el resultado, sino mantenerse constante para que la comparación
# entre escuelas sea justa. Si una variable de este bloque resulta importante,
# eso NO responde la pregunta de investigación: la confirma como condición.
# =========================================================================
contexto_num = [
    "ive_media",         # % de estudiantes en vulnerabilidad prioritaria (JUNAEB)
    "pct_preferentes",   # composición socioeconómica según SEP
    "n_docentes_total",  # tamaño del establecimiento
]
contexto_cat = [
    "gse",   # grupo socioeconómico (Bajo / Medio bajo)
    "zona",  # urbano / rural
]

# =========================================================================
# BLOQUE PROCESO — la respuesta a "qué caracteriza"
#
# Rasgos del funcionamiento del establecimiento. La contribución incremental
# de este bloque, POR SOBRE el de contexto, es la evidencia que responde la
# pregunta de investigación.
# =========================================================================
proceso_num = [
    # Clima escolar reportado por estudiantes (IDPS)
    "autoestima_motiv", "convivencia", "participacion", "vida_saludable",
    # Indicadores construidos por análisis factorial policórico en el EDA
    "promedio_estandarizado_genero",      # creencias de género estereotipadas
    "promedio_estandarizado_clima",       # clima laboral entre docentes
    "promedio_estandarizado_directivos",  # coordinación con el equipo directivo
    # Proceso escolar
    "asistencia_2m",
    # Composición del cuerpo docente
    "pct_tramo_avanzado_sup", "pct_tramo_acceso_inicial",
    "pct_anios_0a5", "pct_anios_21mas",
]
proceso_cat = [
    "depe",  # dependencia administrativa (Municipal / Part. subv. / SLEP)
]

# --- Ensamblaje ---
predictores_num = contexto_num + proceso_num
predictores_cat = contexto_cat + proceso_cat
predictores = predictores_num + predictores_cat

# Diccionario de bloques: se usará en el diseño incremental de modelos
BLOQUES = {
    "contexto": contexto_num + contexto_cat,
    "proceso": proceso_num + proceso_cat,
}

X = df[predictores].copy()
y = df["target_bin"].copy()
grupos = df["cod_com_rbd"].copy()   # para la validación cruzada agrupada

print(f"X: {X.shape[0]} establecimientos x {X.shape[1]} predictores")
print(f"  Bloque CONTEXTO: {len(BLOQUES['contexto']):2d} variables  -> define 'condiciones similares'")
print(f"  Bloque PROCESO:  {len(BLOQUES['proceso']):2d} variables  -> responde 'qué caracteriza'")
print(f"\ny: {y.sum()} positivos / {len(y)} ({y.mean() * 100:.1f}%)")
print(f"grupos: {grupos.nunique()} comunas\n")

# Control de integridad: ninguna variable puede quedar en los dos bloques ni fuera de ambos
assert set(BLOQUES["contexto"]) & set(BLOQUES["proceso"]) == set(), "Hay variables en ambos bloques"
assert set(BLOQUES["contexto"]) | set(BLOQUES["proceso"]) == set(predictores), "Hay variables sin bloque"
print("Bloques disjuntos y exhaustivos: OK\n")

faltantes = X.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("Faltantes que resolverá el pipeline (NO se imputan aquí):")
print(pd.DataFrame({"n": faltantes, "pct": (faltantes / len(X) * 100).round(1)}).to_string())

Nota sobre los faltantes: **no los imputamos en esta celda a propósito.** `X` queda con `NaN`.

Si calculáramos la mediana ahora, sobre las 1.525 filas, esa mediana contendría información de las filas que después usaremos como validación. Es una fuga sutil pero real, y una de las causas más comunes de resultados optimistas que no se replican fuera del notebook. La imputación tiene que ocurrir **dentro del pipeline**, para que en cada partición se ajuste solo con los datos de entrenamiento de esa partición. Lo construimos en el bloque siguiente.

## 5. Preprocesamiento y protocolo de validación

### Cómo la rúbrica condiciona el diseño

La rúbrica pide *"histograma de scores y umbral de decisión, métricas de efectividad, revisión de casos"*, y le asigna dos tercios del puntaje de esta sección. Eso tiene una consecuencia técnica: el histograma de scores y la revisión de casos necesitan **un score fuera de muestra para cada establecimiento**.

Si reserváramos un hold-out del 20%, tendríamos solo 305 establecimientos con score honesto y unos 71 positivos. Es muy poco para construir un histograma informativo, elegir un umbral con criterio y revisar casos con sustancia.

La alternativa que adoptamos es **validación cruzada con predicciones fuera de fold** (`cross_val_predict`): cada establecimiento recibe un score generado por un modelo que **no lo vio durante su entrenamiento**. Son 1.525 scores honestos en lugar de 305, sin sacrificar rigor.

Queda un punto fino de honestidad que abordaremos al elegir el umbral: si se selecciona el umbral óptimo mirando esos mismos scores, la métrica resultante es levemente optimista. Se resuelve eligiendo el umbral dentro de cada fold y aplicándolo a su validación, y reportando ambas cifras.

> Si el curso exige explícitamente una partición train/test, agregarla sobre esta estructura es directo: se reserva el 20% agrupado por comuna y se confirma el resultado final. Avísame y lo incorporo.

### 5.1 Pipeline de preprocesamiento

Tres transformaciones, todas **dentro** del pipeline:

| Paso | Qué hace | Por qué dentro del pipeline |
| :--- | :--- | :--- |
| `SimpleImputer(strategy="median", add_indicator=True)` | Rellena `NaN` con la mediana y agrega una columna binaria que marca dónde faltaba | La mediana debe calcularse solo con el entrenamiento de cada fold |
| `StandardScaler` | Centra y escala las numéricas | La media y desviación también son parámetros que se aprenden |
| `OneHotEncoder(drop="first")` | Convierte las nominales en indicadores, con categoría de referencia | Las categorías presentes pueden variar entre folds |

El `add_indicator=True` es la implementación de la decisión que tomaste sobre los faltantes: en vez de solo inventar un valor, el modelo recibe además la información de que **el dato faltaba**. Si los establecimientos donde los docentes no responden el cuestionario difieren sistemáticamente de los demás, el modelo puede usarlo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


def construir_preprocesador(cols_num, cols_cat):
    """
    Devuelve el preprocesador para un conjunto dado de columnas.

    Se entrega SIN ajustar, para que forme parte del Pipeline y se ajuste dentro
    de cada fold: la mediana de imputación y la media/desviación de la
    estandarización se calculan solo con el entrenamiento de ese fold. Es lo que
    evita la fuga de información en el preprocesamiento.

    Recibe las columnas como argumento porque el diseño incremental necesita
    construir el mismo preprocesamiento para distintos subconjuntos de variables
    (solo contexto, contexto + proceso).
    """
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                # add_indicator=True agrega una columna binaria "faltaba" por cada
                # variable con NaN. Así el modelo puede usar la no respuesta como
                # señal, en vez de solo recibir un valor inventado.
                ("imputar", SimpleImputer(strategy="median", add_indicator=True)),
                ("escalar", StandardScaler()),
            ]), cols_num),
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cols_cat),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


# Inspección: qué columnas produce el preprocesamiento sobre el set completo.
# Se ajusta aquí SOLO para mirar los nombres; este objeto no se reutiliza.
_inspector = construir_preprocesador(predictores_num, predictores_cat).fit(X)
nombres_features = _inspector.get_feature_names_out()

print(f"{X.shape[1]} predictores de entrada -> {len(nombres_features)} columnas de salida\n")
print("Columnas generadas:")
for n in nombres_features:
    marca = "  (indicador de faltante)" if "missingindicator" in n else ""
    print(f"  {n}{marca}")

### 5.2 Validación cruzada agrupada por comuna

Los establecimientos de una misma comuna no son observaciones independientes: comparten mercado educativo, sostenedor y contexto territorial. Si una escuela de Renca queda en entrenamiento y otra de Renca en validación, el modelo puede acertar **por haber visto su vecindario**, no por haber aprendido un mecanismo. El desempeño estimado sería optimista.

`StratifiedGroupKFold` lo evita: mantiene todos los establecimientos de una comuna en el mismo lado de la partición, mientras intenta preservar la proporción de clases. La estratificación es aproximada, porque 171 de las 319 comunas no tienen ningún establecimiento positivo y 88 tienen una sola escuela.

Antes de adoptarlo, lo comparamos contra la validación sin agrupar. La diferencia entre ambos es un diagnóstico interesante en sí mismo: **es la porción del desempeño que provenía de memorizar territorio.**

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict

N_SPLITS = 5

# Protocolo principal: los establecimientos de una misma comuna nunca se reparten
# entre entrenamiento y validación.
cv_agrupada = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Protocolo de contraste, solo para el diagnóstico de abajo.
cv_simple = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Modelo de referencia para todos los diagnósticos de protocolo: regresión
# logística con regularización L2 y pesos balanceados por el desbalance de clases.
modelo_referencia = Pipeline([
    ("pre", construir_preprocesador(predictores_num, predictores_cat)),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)),
])

# --- Diagnóstico: ¿cuánto del desempeño era memorizar territorio? ---
auc_simple = cross_val_score(modelo_referencia, X, y, cv=cv_simple, scoring="roc_auc")
auc_agrupada = cross_val_score(modelo_referencia, X, y, cv=cv_agrupada,
                               groups=grupos, scoring="roc_auc")

print(f"StratifiedKFold      (sin agrupar): ROC-AUC = {auc_simple.mean():.3f} (±{auc_simple.std():.3f})")
print(f"StratifiedGroupKFold (por comuna) : ROC-AUC = {auc_agrupada.mean():.3f} (±{auc_agrupada.std():.3f})")
print(f"Brecha (sin agrupar - agrupada)   : {auc_simple.mean() - auc_agrupada.mean():+.3f}")

# --- Verificación de que la agrupación se respeta de verdad ---
for i, (idx_tr, idx_va) in enumerate(cv_agrupada.split(X, y, groups=grupos)):
    compartidas = set(grupos.iloc[idx_tr]) & set(grupos.iloc[idx_va])
    assert not compartidas, f"Fold {i}: {len(compartidas)} comunas en train y validación"
print(f"\nNinguna comuna aparece a la vez en entrenamiento y validación (en los {N_SPLITS} folds): OK")

**La brecha es prácticamente nula: +0,002.** El ROC-AUC es equivalente agrupando y sin agrupar. Eso significa que el modelo **no está aprendiendo geografía**: su desempeño no depende de haber visto otras escuelas de la misma comuna.

Es un resultado que vale reportar, porque descarta de entrada una crítica evidente sobre la independencia de las observaciones. Nótese, eso sí, que la desviación entre folds sí aumenta (±0,033 contra ±0,013): al agrupar, cada fold es más heterogéneo y la estimación es más variable. Esa mayor incertidumbre es honesta, no un defecto.

Mantenemos la validación agrupada como protocolo principal por dos razones: es la más conservadora, y ahora tenemos evidencia de que **no cuesta desempeño**. Cuando un control de robustez sale gratis, se conserva.

### 5.3 Elección de la métrica principal

Con 23,4% de prevalencia, la elección de métrica no es un detalle técnico: determina si tus conclusiones son válidas. Antes de comparar modelos hay que establecer **contra qué piso** se comparan.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score

# --- Pisos teóricos, dada la prevalencia ---
print(f"Prevalencia de la clase positiva: {y.mean() * 100:.1f}%\n")
print("Piso de cada métrica:")
print(f"  accuracy de 'predecir siempre Insuficiente' : {1 - y.mean():.3f}")
print(f"  PR-AUC de un clasificador aleatorio         : {y.mean():.3f}   (= la prevalencia)")
print(f"  ROC-AUC de un clasificador aleatorio        : 0.500\n")

# --- Los mismos pisos, medidos empíricamente con el protocolo real ---
modelos_piso = {
    "Trivial: siempre 'Insuficiente'": DummyClassifier(strategy="most_frequent"),
    "Aleatorio estratificado": DummyClassifier(strategy="stratified", random_state=SEED),
    "Referencia: log. reg. (18 predictores)": modelo_referencia,
}

filas = []
for nombre, clf in modelos_piso.items():
    score = cross_val_predict(clf, X, y, cv=cv_agrupada, groups=grupos,
                              method="predict_proba")[:, 1]
    filas.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y, (score >= 0.5).astype(int)),
        "ROC-AUC": roc_auc_score(y, score),
        "PR-AUC": average_precision_score(y, score),
    })

pd.DataFrame(filas).set_index("modelo").round(3)

**Lectura de los pisos.** Esta tabla es, por sí sola, el argumento completo contra usar *accuracy* en este problema.

El clasificador trivial — el que **nunca predice la clase positiva** — obtiene **0,766 de accuracy**. No identifica ni un solo establecimiento que supere el corte, es completamente inútil para tu pregunta de investigación, y sin embargo parece acertar tres de cada cuatro veces.

Y el resultado decisivo: el modelo con los 18 predictores tiene **menos accuracy que el trivial** (0,758 vs 0,766), mientras casi **triplica el PR-AUC** (0,617 vs 0,234). No es un error: al usar `class_weight="balanced"`, el modelo predice la clase positiva más seguido, y eso le cuesta aciertos en la clase mayoritaria a cambio de capacidad real de discriminar. Si hubieras elegido accuracy como métrica, habrías concluido que el modelo con información **es peor que no tener modelo**.

El ROC-AUC no sufre ese problema (el trivial queda en 0,500), pero tiene otro: su piso es 0,500 **sea cual sea la prevalencia**, así que no informa cuán difícil es el problema. El PR-AUC sí: su piso **es** la prevalencia (0,234). Cuando reportes PR-AUC de 0,617, la comparación relevante es contra 0,234, y la mejora se interpreta directamente como capacidad de encontrar establecimientos positivos sin inundarse de falsos positivos.

**Decisión de métricas:**

| Métrica | Rol |
| :--- | :--- |
| **PR-AUC** (*average precision*) | Principal. Para seleccionar modelos y comparar bloques. |
| **ROC-AUC** | Secundaria. Comparable con los valores reportados en el EDA (0,738–0,829). |
| **Precisión y recall de la clase positiva** | Al umbral elegido. Base de la revisión de casos. |
| **Matriz de confusión** | Al umbral elegido, para cuantificar los dos tipos de error por separado. |
| *Accuracy* | Se reporta únicamente para documentar por qué no sirve aquí. |

In [ ]:
# Contenedores para reutilizar los resultados en las secciones siguientes
scores_oof = {}                 # score fuera de fold, uno por establecimiento
hiperparametros_por_fold = {}   # trazabilidad del ajuste

filas = []
for nombre, (clasificador, grilla) in espacio_modelos.items():
    t0 = time.time()
    oof, elegidos = evaluar_anidado(
        nombre,
        construir_preprocesador(predictores_num, predictores_cat),
        clasificador,
        grilla,
    )
    scores_oof[nombre] = oof
    hiperparametros_por_fold[nombre] = elegidos

    filas.append({
        "modelo": nombre,
        "PR-AUC": average_precision_score(y, oof),
        "ROC-AUC": roc_auc_score(y, oof),
        "combinaciones": int(np.prod([len(v) for v in grilla.values()])),
        "segundos": round(time.time() - t0, 1),
    })

resultados = pd.DataFrame(filas).set_index("modelo").sort_values("PR-AUC", ascending=False)
print(f"Piso del PR-AUC (prevalencia): {y.mean():.3f}\n")
print("Desempeño en validación cruzada anidada y agrupada:")
print(resultados.round(3).to_string())

print("\n\nHiperparámetros elegidos en cada fold externo (trazabilidad del ajuste):")
for nombre, elegidos in hiperparametros_por_fold.items():
    print(f"\n  {nombre}")
    for i, e in enumerate(elegidos):
        print(f"    fold {i}: {e}")

**Lectura de la comparación.**

**Gana la regresión logística** (PR-AUC 0,618), por delante de Random Forest (0,581), HistGB (0,537) y muy por delante del árbol simple (0,471). Vale detenerse en esto, porque contradice la creencia habitual de que el *gradient boosting* siempre gana en datos tabulares:

> Esa creencia vale para muestras grandes. Con **1.525 filas y 22 columnas**, los ensembles tienen capacidad de sobra para memorizar y poca información nueva que extraer. Las relaciones aquí son mayormente monótonas y suaves — más IDPS, más probabilidad de superar el corte — que es exactamente el escenario donde un modelo lineal regularizado es difícil de superar. Los árboles gastan flexibilidad en modelar quiebres que no existen.

**El árbol simple es el peor con diferencia** (0,471, apenas el doble del piso de 0,234). Es esperable y útil de mostrar: un solo árbol produce predicciones escalonadas y de alta varianza, y con clases desbalanceadas divide de forma inestable. Sirve como cota inferior de los métodos no lineales.

**Estabilidad de los hiperparámetros.** `C = 0,1` fue elegido en 4 de los 5 folds (y `C = 0,01` en el quinto). Que la búsqueda converja al mismo valor en folds independientes indica que no está capturando ruido. Y el valor tiene lectura sustantiva: `C = 0,1` es una regularización **fuerte**, que es justo lo que corresponde dada la colinealidad que documentamos en la sección 4.2. El modelo está repartiendo el peso entre los IDPS correlacionados en vez de asignarlo arbitrariamente a uno.

El Random Forest eligió `min_samples_leaf = 10` en los 5 folds y `max_features = 'sqrt'` en los 5: también converge, hacia más regularización. Ambos modelos coinciden en que estos datos **piden restricción, no flexibilidad**.

### 6.3 Dos verificaciones adicionales

Antes de fijar el modelo principal, dos comprobaciones que ponen a prueba decisiones anteriores.

In [ ]:
from sklearn.linear_model import LogisticRegression

# --- Variante A: HistGB usando NaN de forma nativa, sin imputar ni escalar ---
# Prueba directa de la decisión de imputación: los árboles de HistGB pueden
# enviar los faltantes a la rama que minimice la pérdida, en vez de recibir
# la mediana. Si esta variante ganara claramente, habría que revisar la decisión.
preprocesador_nan = ColumnTransformer(
    transformers=[
        ("num", "passthrough", predictores_num),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), predictores_cat),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# --- Variantes B y C: otras penalizaciones para la regresión logística ---
# Si el resultado cambiara mucho según la penalización, la señal sería frágil.
variantes = {
    "HistGB (NaN nativo)": (
        preprocesador_nan,
        HistGradientBoostingClassifier(max_iter=300, early_stopping=True,
                                       validation_fraction=0.15,
                                       class_weight="balanced", random_state=SEED),
        {"clf__learning_rate": [0.05, 0.1], "clf__max_leaf_nodes": [15, 31],
         "clf__min_samples_leaf": [10, 20]},
    ),
    "Log. L1 (selección)": (
        construir_preprocesador(predictores_num, predictores_cat),
        LogisticRegression(penalty="l1", solver="saga", max_iter=5000,
                           class_weight="balanced", random_state=SEED),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "Log. ElasticNet": (
        construir_preprocesador(predictores_num, predictores_cat),
        LogisticRegression(penalty="elasticnet", solver="saga", max_iter=5000,
                           class_weight="balanced", random_state=SEED),
        {"clf__C": [0.1, 1], "clf__l1_ratio": [0.2, 0.5, 0.8]},
    ),
}

filas_var = []
for nombre, (prep, clf, grilla) in variantes.items():
    oof, elegidos = evaluar_anidado(nombre, prep, clf, grilla)
    scores_oof[nombre] = oof
    hiperparametros_por_fold[nombre] = elegidos
    filas_var.append({
        "modelo": nombre,
        "PR-AUC": average_precision_score(y, oof),
        "ROC-AUC": roc_auc_score(y, oof),
    })

print("\nVariantes de robustez (comparar contra Log. reg. L2 = 0.618):")
pd.DataFrame(filas_var).set_index("modelo").round(3)

### 6.4 Modelo principal

Las tres variantes de penalización rinden lo mismo (0,618–0,620): el resultado **no depende de la elección de penalización**, lo que es evidencia de que la señal es robusta y no un artefacto de una configuración particular.

Se adopta la **regresión logística con penalización L2 y `C = 0,1`** como modelo principal, por tres razones:

1. **Rinde igual o mejor que todo lo demás** (PR-AUC 0,618 en CV anidada agrupada).
2. **L2 conserva todos los coeficientes**, mientras L1 anula algunos. Para la sección de caracterización necesitamos ver el aporte de todas las variables de proceso, no solo de las que sobreviven a una selección automática.
3. **Es el modelo más interpretable disponible**, lo que en este proyecto no es una concesión: la pregunta de investigación es "qué caracteriza", no "cuánto se puede predecir".

Los modelos de árboles se conservan en el notebook como evidencia comparativa. Que un modelo lineal regularizado gane a los ensembles **es** un resultado reportable, y muestra que la búsqueda de hiperparámetros se hizo en serio en vez de asumir que el boosting gana siempre.

> **Nota sobre el ajuste de hiperparámetros y la rúbrica:** el diseño anidado es lo que hace legítima la frase "ajustamos hiperparámetros". Si eligiéramos `C` mirando el desempeño en los mismos datos con que después reportamos la métrica, el número estaría contaminado. La tabla de hiperparámetros por fold documenta el proceso y, además, muestra su estabilidad.

---

*(Continúa: depuración de las variables candidatas, protocolo de validación, modelos y evaluación.)*